In [1]:
import numpy as np
import pandas as pd
import os
import lightgbm as lgb

os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

In [ ]:
X = pd.read_csv('data/final/feature.csv')
y = pd.read_csv('data/final/target.csv')
y = y[['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON', 'TARGET_LAT_12h', 'TARGET_LON_12h']]

df = pd.merge(X, y, on=['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON'], how='left')

In [3]:
# ==========================================
# PHẦN 1: HÀM ĐÁNH GIÁ SAI SỐ HAVERSINE
# ==========================================
def haversine_distance(lat_true, lon_true, lat_pred, lon_pred):
    """Tính khoảng cách thực tế (km) trên mặt cầu Trái Đất"""
    lat_true, lon_true, lat_pred, lon_pred = map(np.radians, [lat_true, lon_true, lat_pred, lon_pred])
    dlat = lat_pred - lat_true
    dlon = lon_pred - lon_true
    a = np.sin(dlat/2.0)**2 + np.cos(lat_true) * np.cos(lat_pred) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # Bán kính Trái Đất R = 6371 km

In [4]:
# ==========================================
# PHẦN 2: CHUẨN BỊ VÀ CHIA TẬP DỮ LIỆU (TIME-BASED SPLIT)
# ==========================================
# Xóa các dòng có Target bị rỗng (NaN) khi bão kết thúc
df_clean = df.dropna(subset=['TARGET_LAT_12h', 'TARGET_LON_12h']).copy() 

# Chia tập dữ liệu dựa trên mốc thời gian (Năm) để chống Data Leakage tuyệt đối
train_df = df_clean[(df_clean['YEAR'] >= 2000) & (df_clean['YEAR'] <= 2020)]  # 21 years
val_df = df_clean[(df_clean['YEAR'] >= 2021) & (df_clean['YEAR'] <= 2022)]    # 2 years
test_df = df_clean[(df_clean['YEAR'] >= 2023) & (df_clean['YEAR'] <= 2024)]   # 2 years

# Định nghĩa các cột không phải là Feature (như mã bão, thời gian, tọa độ hiện tại và target)
drop_cols = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'TARGET_LAT_12h', 'TARGET_LON_12h']

# Tách biến độc lập (X) và biến phụ thuộc (y)
X_train = train_df.drop(columns=drop_cols)
y_train_lat = train_df['TARGET_LAT_12h']
y_train_lon = train_df['TARGET_LON_12h']

X_val = val_df.drop(columns=drop_cols)
y_val_lat = val_df['TARGET_LAT_12h']
y_val_lon = val_df['TARGET_LON_12h']

X_test = test_df.drop(columns=drop_cols)
y_test_lat = test_df['TARGET_LAT_12h']
y_test_lon = test_df['TARGET_LON_12h']

# Giữ lại tọa độ hiện tại (T0) của tập Test để tính ngược ra vị trí tương lai trên bản đồ
current_lat_test = test_df['LAT'].values
current_lon_test = test_df['LON'].values

In [5]:
# ==========================================
# PHẦN 3: KHỞI TẠO VÀ HUẤN LUYỆN MÔ HÌNH GBDT
# ==========================================
# Cấu hình siêu tham số chuẩn theo nghiên cứu của Jinkai Tan để tránh Over-fitting
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    'learning_rate': 0.02,      # Tốc độ học nhỏ
    'n_estimators': 3000,       # Số cây lớn để early stopping hoạt động
    'max_depth': 8,             # Theo cấu hình Jinkai Tan
    'num_leaves': 7,            # Nhỏ để tránh overfitting
    'feature_fraction': 0.6,    # Lấy 70% features mỗi lần fit cây
    'lambda_l1': 0.1,           # Regularization
    'lambda_l2': 1,
}

print("Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...")
model_lat = lgb.LGBMRegressor(**lgb_params)
model_lat.fit(
    X_train, y_train_lat,
    eval_set=[(X_val, y_val_lat)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print("Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...")
model_lon = lgb.LGBMRegressor(**lgb_params)
model_lon.fit(
    X_train, y_train_lon,
    eval_set=[(X_val, y_val_lon)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[579]	valid_0's rmse: 0.498378
Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[674]	valid_0's rmse: 0.612773


,boosting_type,'gbdt'
,num_leaves,7
,max_depth,8
,learning_rate,0.02
,n_estimators,3000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [6]:
# ==========================================
# PHẦN 4: DỰ BÁO VÀ ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
# 1. Dự báo mức độ dịch chuyển (Delta Lat, Delta Lon)
pred_delta_lat = model_lat.predict(X_test)
pred_delta_lon = model_lon.predict(X_test)

# 2. Cộng mức độ dịch chuyển vào tọa độ T0 để ra tọa độ dự báo thực tế tại T+24h
pred_lat_24h = current_lat_test + pred_delta_lat
pred_lon_24h = current_lon_test + pred_delta_lon

true_lat_24h = current_lat_test + y_test_lat.values
true_lon_24h = current_lon_test + y_test_lon.values

# 3. Tính toán sai số khoảng cách bằng Haversine
test_errors_km = haversine_distance(true_lat_24h, true_lon_24h, pred_lat_24h, pred_lon_24h)
mean_error = np.mean(test_errors_km)

print(f"\n=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 24H TRUNG BÌNH (Mean Forecast Error): {mean_error:.2f} km")


=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 24H TRUNG BÌNH (Mean Forecast Error): 62.75 km


In [7]:
import numpy as np
import pandas as pd

def evaluate_advanced_metrics(y_true, y_pred, v_past):
    """
    y_true: Mảng chứa [delta_lat_obs, delta_lon_obs]
    y_pred: Mảng chứa [delta_lat_pred, delta_lon_pred]
    v_past: Vận tốc bão trong 24h qua (V_Actual_24h từ dữ liệu gốc)
    """
    
    # Chuyển đổi sang numpy array để tính toán vector nhanh
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    v_past = np.array(v_past)
    
    # Tách biệt Delta Lat (Y) và Delta Lon (X)
    dY_obs, dX_obs = y_true[:, 0], y_true[:, 1]
    dY_pred, dX_pred = y_pred[:, 0], y_pred[:, 1]
    
    # ---------------------------------------------------------
    # 2. DIRECTIONAL STABILITY (Độ ổn định hướng)
    # ---------------------------------------------------------
    # Điều kiện đúng: Cùng dấu (Tích > 0) cho cả vĩ độ và kinh độ
    correct_direction = (dX_obs * dX_pred > 0) & (dY_obs * dY_pred > 0)
    directional_stability = np.mean(correct_direction) * 100
    
    # ---------------------------------------------------------
    # 3. MOVING VELOCITY SENSITIVITY (Độ nhạy vận tốc)
    # ---------------------------------------------------------
    # Tính vận tốc dự báo và thực tế trong 24h tới (V_next)
    # Công thức: sqrt(delta_lat^2 + delta_lon^2) / thời gian
    # Ở đây dùng đơn vị tương đối (quãng đường dịch chuyển) để so sánh tỷ lệ
    v_next_obs = np.sqrt(dX_obs**2 + dY_obs**2)
    v_next_pred = np.sqrt(dX_pred**2 + dY_pred**2)
    
    # Tránh lỗi chia cho 0 nếu bão đứng yên
    v_past = np.where(v_past == 0, 0.001, v_past)
    
    # Xác định các ca biến động mạnh (Ground Truth)
    accel_cases = (v_next_obs / v_past > 2.0)
    decel_cases = (v_next_obs / v_past < 0.5)
    
    # Kiểm tra mô hình có bắt trúng không
    accel_hit = accel_cases & (v_next_pred / v_past > 2.0)
    decel_hit = decel_cases & (v_next_pred / v_past < 0.5)
    
    # Tính tỷ lệ nhạy (Sensitivity)
    total_anomalies = np.sum(accel_cases) + np.sum(decel_cases)
    total_hits = np.sum(accel_hit) + np.sum(decel_hit)
    
    # Tránh chia cho 0 nếu tập test không có ca đột biến nào
    velocity_sensitivity = (total_hits / total_anomalies * 100) if total_anomalies > 0 else 0
    
    # In kết quả
    print("--- KẾT QUẢ ĐÁNH GIÁ CHUYÊN SÂU ---")
    print(f"1. Directional Stability: {directional_stability:.2f}%")
    print(f"   (Kỳ vọng: > 87%)")
    print(f"2. Moving Velocity Sensitivity: {velocity_sensitivity:.2f}%")
    print(f"   (Kỳ vọng: ~ 40-60% - Đây là điểm yếu của GBDT)")
    print(f"   - Số ca đột biến thực tế: {total_anomalies}")
    print(f"   - Số ca mô hình bắt trúng: {total_hits}")

    return directional_stability, velocity_sensitivity


v_past = test_df['V_Actual_12h'].values

actual = np.column_stack((y_test_lat, y_test_lon))
y_test_actual = pd.DataFrame(actual)

predict = np.column_stack((pred_delta_lat, pred_delta_lon))
y_test_predicted = pd.DataFrame(predict)

evaluate_advanced_metrics(y_test_actual, y_test_predicted, v_past)

--- KẾT QUẢ ĐÁNH GIÁ CHUYÊN SÂU ---
1. Directional Stability: 85.83%
   (Kỳ vọng: > 87%)
2. Moving Velocity Sensitivity: 10.42%
   (Kỳ vọng: ~ 40-60% - Đây là điểm yếu của GBDT)
   - Số ca đột biến thực tế: 48
   - Số ca mô hình bắt trúng: 5


(np.float64(85.8288770053476), np.float64(10.416666666666668))